<a href="https://colab.research.google.com/github/Siddtiwa/Karpathy-assignments/blob/main/Bigram_Model_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
#Imports
from google.colab import files
import torch
import torch.nn.functional as F

In [4]:
uploaded = files.upload()

Saving names.txt to names.txt


In [5]:
words = open("names.txt", "r").read().splitlines()

In [30]:
words[:10]

['emma',
 'olivia',
 'ava',
 'isabella',
 'sophia',
 'charlotte',
 'mia',
 'amelia',
 'harper',
 'evelyn']

In [31]:
vocab = ""

for word in words:
  vocab += word

vocab = sorted(list(set(vocab)))
vocab

['a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'j',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'q',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'x',
 'y',
 'z']

In [191]:
# index to string
itos = {i+1:c for i, c in enumerate(vocab)}
itos[0] = "."
itos

{1: 'a',
 2: 'b',
 3: 'c',
 4: 'd',
 5: 'e',
 6: 'f',
 7: 'g',
 8: 'h',
 9: 'i',
 10: 'j',
 11: 'k',
 12: 'l',
 13: 'm',
 14: 'n',
 15: 'o',
 16: 'p',
 17: 'q',
 18: 'r',
 19: 's',
 20: 't',
 21: 'u',
 22: 'v',
 23: 'w',
 24: 'x',
 25: 'y',
 26: 'z',
 0: '.'}

In [192]:
# string to index
stoi = {c:i for i, c in itos.items()}
stoi

{'a': 1,
 'b': 2,
 'c': 3,
 'd': 4,
 'e': 5,
 'f': 6,
 'g': 7,
 'h': 8,
 'i': 9,
 'j': 10,
 'k': 11,
 'l': 12,
 'm': 13,
 'n': 14,
 'o': 15,
 'p': 16,
 'q': 17,
 'r': 18,
 's': 19,
 't': 20,
 'u': 21,
 'v': 22,
 'w': 23,
 'x': 24,
 'y': 25,
 'z': 26,
 '.': 0}

In [193]:
xs, ys = [], []
for word in words:
  chs = ["."] + list(word) + ["."]
  for ch1, ch2 in zip(chs, chs[1:]):
    ix1 = stoi[ch1]
    ix2 = stoi[ch2]
    xs.append(ix1)
    ys.append(ix2)
print(len(xs), len(ys))

228146 228146


In [194]:
# Train - test split
xs_train_split = xs[:int(len(xs)*0.8)]
ys_train_split = ys[:int(len(ys)*0.8)]
xs_val_split = xs[int(len(xs)*0.8):]
ys_val_split = ys[int(len(ys)*0.8):]

In [195]:
xs_train = torch.tensor(xs_train_split)
ys_train = torch.tensor(ys_train_split)
xs_val = torch.tensor(xs_val_split)
ys_val = torch.tensor(ys_val_split)
num = xs_train.nelement()
xs_train[:10], ys_train[:10]

(tensor([ 0,  5, 13, 13,  1,  0, 15, 12,  9, 22]),
 tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9]))

In [196]:
# Creating layer
g = torch.Generator().manual_seed(2147483647)
w = torch.randn((27,27), generator = g, requires_grad=True)

In [233]:
xenc = F.one_hot(xs_train, num_classes=27).float() # ohe the input data
logits = xenc @ w # w.x + b
count = logits.exp()
prob = count / count.sum(1, keepdims=True) # softmax
loss = -prob[torch.arange(num), ys_train].log().mean() + 0.01*(w**2).mean() # loss with regularization
loss

tensor(2.5673, grad_fn=<AddBackward0>)

In [234]:
# training
w.grad = None
loss.backward()
w.data += -50.0 * w.grad # updating weights

In [235]:
# Sampling from training data
g = torch.Generator().manual_seed(2147483647)
for i in range(5):
  out = []
  ix = 0

  while True:
    xenc = F.one_hot(torch.tensor([ix]), num_classes = 27).float()
    logits = xenc @ w
    count = logits.exp()
    p = count/count.sum(1, keepdims=True)
    ix = torch.multinomial(p, num_samples=1, replacement=True, generator=g).item()
    out.append(itos[ix])
    if ix == 0:
      break
  print("".join(out))

cexza.
memalurailezityha.
minimittain.
llayn.
ka.


In [236]:
# Calculating loss for test data
g = torch.Generator().manual_seed(2147483647)
num = xs_val.nelement()
xenct = F.one_hot(xs_val, num_classes = 27).float()
logits = xenct @ w
count = logits.exp()
prob = count/count.sum(1, keepdims=True)
loss = -prob[torch.arange(num), ys_val].log().mean() + 0.01*(w**2).mean()
loss

tensor(2.7364, grad_fn=<AddBackward0>)